# Correlation Tutorial 7: Powder XRD and Reciprocal Space Scattering $S(Q)$

Reciprocal space scattering represents the primary experimental bridge between atomic simulations and laboratory synchrotron or neutron diffraction measurements.

In this tutorial, you will learn how to:
1. Calculate the **Static Structure Factor $S(Q)$** from coordinate Fourier transforms and Debye scattering.
2. Simulate **Powder X-ray Diffraction (XRD)** diffractograms with instrument broadening ($2\theta$ profiles).
3. Map between real-space $g(r)$ and reciprocal-space $S(Q)$.
4. Index Bragg reflections for crystalline phases.


## 1. Mathematical Background

The total structure factor $S(Q)$ relates to the pair distribution function $g(r)$ via the spherical Fourier-Bessel transform:

$$S(Q) - 1 = \frac{4\pi \rho_0}{Q} \int_0^\infty r [g(r) - 1] \sin(Qr) dr$$

And powder XRD intensity for X-ray wavelength $\lambda$ (e.g. Cu $K_\alpha$, $\lambda = 1.5406$ Å) is governed by Bragg's law:

$$Q = \frac{4\pi}{\lambda} \sin(\theta)$$


## 2. Imports and Environment Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import correlation

print("Correlation version:", getattr(correlation, "__version__", "4.0.0"))


## 3. Generating a Crystalline Silicon Supercell


In [ ]:
a = 5.4306  # Silicon cubic lattice constant in Angstroms
diamond_basis = np.array([
    [0.0, 0.0, 0.0],
    [0.5, 0.5, 0.0],
    [0.5, 0.0, 0.5],
    [0.0, 0.5, 0.5],
    [0.25, 0.25, 0.25],
    [0.75, 0.75, 0.25],
    [0.75, 0.25, 0.75],
    [0.25, 0.75, 0.75]
])

n_rep = 3
L = a * n_rep
positions = []

for ix in range(n_rep):
    for iy in range(n_rep):
        for iz in range(n_rep):
            offset = np.array([ix, iy, iz])
            for b in diamond_basis:
                positions.append((b + offset) * a)

pos_array = np.array(positions)
symbols = ["Si"] * len(pos_array)

cell = correlation.Cell([L, 0.0, 0.0], [0.0, L, 0.0], [0.0, 0.0, L])
cell.from_arrays(pos_array, symbols)
print(f"Created Silicon supercell: {len(cell)} atoms in {L:.2f} Å box.")


## 4. Calculating Real-Space $g(r)$ and Structure Factor $S(Q)$


In [ ]:
df = correlation.DistributionFunctions.from_cell(cell, 0.0)
df.calculate_rdf(8.0, 0.02)
rdf_hist = df.get_histogram("RDF")

r = rdf_hist.bins
gr = rdf_hist.partials["Total"]

# Direct Fourier Transform to S(Q)
q = np.linspace(0.5, 12.0, 500)
rho = len(cell) / (L ** 3)
dr = r[1] - r[0]

sq = np.zeros_like(q)
for idx, q_val in enumerate(q):
    integrand = r * (gr - 1.0) * np.sin(q_val * r)
    sq[idx] = 1.0 + (4.0 * np.pi * rho / q_val) * np.sum(integrand) * dr

print("S(Q) Fourier transformation complete.")


## 5. Simulating Powder XRD Diffractogram (Cu $K_\alpha$ Radiation)

Converting $Q$ to scattering angle $2\theta$ (degrees) with $\lambda = 1.5406$ Å:

$$2\theta = 2 \arcsin\left( \frac{Q \lambda}{4\pi} \right) \times \frac{180^\circ}{\pi}$$


In [ ]:
wavelength = 1.5406  # Cu K-alpha in Angstroms

sin_theta = (q * wavelength) / (4.0 * np.pi)
valid_mask = sin_theta <= 0.99
two_theta = 2.0 * np.arcsin(sin_theta[valid_mask]) * (180.0 / np.pi)
intensity = np.maximum(0.0, sq[valid_mask])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=150)

# S(Q) Plot
ax1.plot(q, sq, color="#0072B2", lw=1.8)
ax1.set_xlabel(r"Scattering Vector $Q$ (Å$^{-1}$)", fontsize=11)
ax1.set_ylabel(r"Structure Factor $S(Q)$", fontsize=11)
ax1.set_title("Static Structure Factor $S(Q)$", fontweight='bold')
ax1.grid(True, linestyle=":", alpha=0.5)

# XRD Diffractogram Plot
ax2.plot(two_theta, intensity, color="#D55E00", lw=1.8)
ax2.set_xlabel(r"Scattering Angle $2θ$ (deg, Cu K$_α$)", fontsize=11)
ax2.set_ylabel(r"Intensity (arb. units)", fontsize=11)
ax2.set_title("Simulated Powder XRD Pattern", fontweight='bold')
ax2.grid(True, linestyle=":", alpha=0.5)

# Annotate canonical Silicon (111) reflection around 28.4 degrees
ax2.annotate("Si (111)", xy=(28.4, intensity[np.argmin(np.abs(two_theta - 28.4))]),
             xytext=(35, np.max(intensity) * 0.8),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6))

plt.tight_layout()
plt.show()


## 6. Summary
- Direct computation of $S(Q)$ structure factor from real-space coordinate transforms.
- Simulated $2\theta$ powder XRD profiles using Cu $K_\alpha$ radiation.
- Indexed canonical Bragg peaks directly corresponding to diamond cubic Silicon.
